# Lab03 — Phần 1. Phân cụm khách hàng

Run All tạo một lượt khách hàng mới bằng cùng pipeline CLI. Thuật toán K-Means, Silhouette và ARI dùng mã NumPy của dự án.

## EDA theo giới tính — tham khảo notebook Kaggle

Phân bố giới tính, histogram và boxplot tuổi/thu nhập/chi tiêu theo giới tính, scatter từng cặp biến và heatmap tương quan Pearson. Số liệu được tính lại từ CSV của dự án; không chép các kết luận của notebook mẫu. Gender trong phần này dùng phân tích mô tả, chưa thay đổi đặc trưng phân cụm.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
eda_repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Lab03/config.yaml").is_file())
if str(eda_repo) not in sys.path: sys.path.insert(0, str(eda_repo))
from Lab03.src.part1_customers.eda import analyze
eda_frame = pd.read_csv(eda_repo / "Lab03/data/customers/raw/Mall_Customers.csv")
eda_output = eda_repo / "Lab03/notebooks/notebook_exports/part1_eda"
analyze(eda_frame, ["Age", "Annual Income (k$)", "Spending Score (1-100)"], eda_output)
print(pd.read_csv(eda_output / "eda/tables/eda_gender_counts.csv").to_string(index=False))
for filename in ["eda_gender_distribution.png", "eda_age_by_gender.png", "eda_income_by_gender.png", "eda_spending_by_gender.png", "eda_income_by_age_gender.png", "eda_feature_pairs_by_gender.png", "eda_correlations.png", "eda_age_spending_joint.png", "eda_age_income_regression.png"]:
    plt.figure(figsize=(15, 8))
    plt.imshow(plt.imread(eda_output / "eda" / filename))
    plt.axis("off")
    plt.show()


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Lab03/config.yaml").is_file())
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from Lab03.src.pipeline import run_pipeline
run = run_pipeline(part="customers")
part = run
print(run)

## Chọn K

Silhouette đo độ tách cụm, inertia đo độ gọn. Cấu hình dùng nhiều lần khởi tạo để giảm ảnh hưởng seed.

In [ ]:
comparison = pd.read_csv(part / "tuning/metrics_by_k.csv")
print(comparison.to_string(index=False))
plt.imshow(plt.imread(part / "figures/silhouette_by_k.png"))
plt.axis("off")
plt.show()


## Mô tả khách hàng và trực quan hóa

In [ ]:
profiles = pd.read_csv(part / "results/cluster_profile.csv")
customers = pd.read_csv(part / "results/customers_labeled.csv")
print(profiles.to_string(index=False))
for name in ("income_spending.png", "silhouette_clusters.png"):
    plt.figure(figsize=(11, 5))
    plt.imshow(plt.imread(part / "figures" / name))
    plt.axis("off")
    plt.show()
print("3D:", part / "figures/customer_clusters_3d.html")


## Độ ổn định và nguồn kết quả

In [ ]:
print(pd.read_csv(part / "results/seed_stability.csv").to_string(index=False))
print(pd.read_csv(part / "results/pairwise_ari.csv").describe().to_string())
print(json.loads((run / "run.json").read_text()))


## Đánh giá lại theo run_id
CLI: python -m Lab03.src.part1_customers.evaluate --run-id <run_id>.
Mô hình được nạp lại để đánh giá cùng tập khách hàng đã phân cụm.

In [ ]:
print(pd.read_csv(part / "results/metrics.csv").to_string(index=False))